## Research Outreach App Upgrade

- **SQLite memory**: `AsyncSqliteSaver` persists checkpoints to `research_memory.db`.
- Use **Get clarifying questions**, then optional answers in **Clarifications** (injected into planner context and worker system prompt).
- **Planning agent**: `START → planner → worker → tools/evaluator` — planner outputs structured steps; worker follows the plan and tools.

In [ ]:

import os
import uuid
import asyncio
import requests

from playwright.async_api import async_playwright
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from dotenv import load_dotenv
from langchain.agents import Tool
from langchain_community.agent_toolkits import FileManagementToolkit
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_experimental.tools import PythonREPLTool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper

from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from typing import List, Any, Optional, Dict
from pydantic import BaseModel, Field
from datetime import datetime

import gradio as gr


In [ ]:
load_dotenv(override=True)
MODEL = "openai/gpt-4o"


True

In [35]:
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter_base_url = os.getenv("OPEN_ROUTER_BASE_URL")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"
serper = GoogleSerperAPIWrapper()

In [36]:
# tools 

async def playwright_tools():
    playwright = await async_playwright().start()
    browser = await playwright.chromium.launch(headless=False)
    toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=browser)
    return toolkit.get_tools(), browser, playwright


def push(text: str):
    """Send a push notification to the user"""
    requests.post(pushover_url, data = {"token": pushover_token, "user": pushover_user, "message": text})
    return "success"


def get_file_tools():
    toolkit = FileManagementToolkit(root_dir="sandbox")
    return toolkit.get_tools()


async def other_tools():
    push_tool = Tool(name="send_push_notification", func=push, description="Use this tool when you want to send a push notification")
    file_tools = get_file_tools()

    tool_search =Tool(
        name="search",
        func=serper.run,
        description="Use this tool when you want to get the results of an online web search"
    )

    wikipedia = WikipediaAPIWrapper()
    wiki_tool = WikipediaQueryRun(api_wrapper=wikipedia)

    python_repl = PythonREPLTool()
    
    return file_tools + [push_tool, tool_search, python_repl,  wiki_tool]

In [37]:
class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool
    plan: Optional[Any]
    final_worker_response: Optional[str]
    clarifications_text: Optional[str]


class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the assistant's response")
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    user_input_needed: bool = Field(
        description="True if more input is needed from the user, or clarifications, or the assistant is stuck"
    )


class ClarifyingQuestions(BaseModel):
    questions: list[str] = Field(
        description="Exactly 3 clarifying questions to narrow the user's request.",
        min_length=3,
        max_length=3,
    )


class PlanOutput(BaseModel):
    list_of_steps: List[str] = Field(description="Ordered steps for the worker to execute")
    estimated_complexity: str = Field(description="Brief estimate of task complexity")


In [ ]:

class Sidekick:
    def __init__(self):
        self.worker_llm_with_tools = None
        self.evaluator_llm_with_output = None
        self.planner_llm = None
        self.tools = None
        self.llm_with_tools = None
        self.graph = None
        self.sidekick_id = str(uuid.uuid4())
        self.memory_ctx = None
        self.memory = None
        self.browser = None
        self.playwright = None

    async def setup(self):
        try:
            if self.memory is None:
                self.memory_ctx = AsyncSqliteSaver.from_conn_string("research_memory.db")
                self.memory = await self.memory_ctx.__aenter__()
            self.tools, self.browser, self.playwright = await playwright_tools()
            self.tools += await other_tools()
        except Exception as e:
            print(f"Setup failed: {e}")
            raise e

            
        worker_llm = ChatOpenAI(model=MODEL, base_url=openrouter_base_url, api_key=openrouter_api_key)
        self.worker_llm_with_tools = worker_llm.bind_tools(self.tools)
        planner_llm = ChatOpenAI(model=MODEL, base_url=openrouter_base_url, api_key=openrouter_api_key)
        self.planner_llm = planner_llm.with_structured_output(PlanOutput)
        evaluator_llm = ChatOpenAI(model=MODEL, base_url=openrouter_base_url, api_key=openrouter_api_key)
        self.evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)
        await self.build_graph()

    def planner(self, state: State) -> Dict[str, Any]:
        """Planning agent: break the request into steps for the worker."""
        conv = self.format_conversation(state["messages"])
        user_msg = f"""Conversation and user request:
{conv}

Success criteria:
{state["success_criteria"]}

Produce an ordered list of concrete steps the worker should follow, and estimate complexity."""
        planner_messages = [
            SystemMessage(
                content="You are a planning agent. Output clear, actionable steps. Respect any user clarifications already in the conversation."
            ),
            HumanMessage(content=user_msg),
        ]
        plan = self.planner_llm.invoke(planner_messages)
        summary = (
            f"Plan generated.\nSteps: {plan.list_of_steps}\n"
            f"Estimated complexity: {plan.estimated_complexity}"
        )
        return {
            "messages": [AIMessage(content=summary)],
            "plan": plan,
        }

    def worker(self, state: State) -> Dict[str, Any]:
        plan = state.get("plan")
        plan_block = ""
        if plan is not None:
            plan_block = f"""
    You must follow this plan in order (delegated by the planning agent):
    Steps: {plan.list_of_steps}
    Estimated complexity: {plan.estimated_complexity}
    """
        clar = state.get("clarifications_text") or ""
        clar_block = ""
        if clar.strip():
            clar_block = f"""
    User clarifications (answers to upfront questions):
    {clar.strip()}
    """
        system_message = f"""You are a helpful assistant that can use tools to complete tasks.
    You keep working on a task until either you have a question or clarification for the user, or the success criteria is met.
    You have many tools to help you, including tools to browse the internet, navigating and retrieving web pages.
    You have a tool to run python code, but note that you would need to include a print() statement if you wanted to receive output.
    The current date and time is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
    {plan_block}
    {clar_block}
    This is the success criteria:
    {state["success_criteria"]}
    You should reply either with a question for the user about this assignment, or with your final response.
    If you have a question for the user, you need to reply by clearly stating your question. An example might be:

    Question: please clarify whether you want a summary or a detailed answer

    If you've finished, reply with the final answer, and don't ask a question; simply reply with the answer.
    """

        if state.get("feedback_on_work"):
            system_message += f"""
    Previously you thought you completed the assignment, but your reply was rejected because the success criteria was not met.
    Here is the feedback on why this was rejected:
    {state["feedback_on_work"]}
    With this feedback, please continue the assignment, ensuring that you meet the success criteria or have a question for the user."""

        # Add in the system message

        found_system_message = False
        messages = state["messages"]
        for message in messages:
            if isinstance(message, SystemMessage):
                message.content = system_message
                found_system_message = True

        if not found_system_message:
            messages = [SystemMessage(content=system_message)] + messages

        # Invoke the LLM with tools
        response = self.worker_llm_with_tools.invoke(messages)
        current = (
            (response.content or "").strip()
            or (state.get("final_worker_response") or "")
        )

        return {
            "messages": [response],
            "final_worker_response": current or None,
        }

    def worker_router(self, state: State) -> str:
        last_message = state["messages"][-1]

        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            return "tools"
        else:
            return "evaluator"

    def format_conversation(self, messages: List[Any]) -> str:
        conversation = "Conversation history:\n\n"
        for message in messages:
            if isinstance(message, HumanMessage):
                conversation += f"User: {message.content}\n"
            elif isinstance(message, AIMessage):
                text = message.content or "[Tools use]"
                conversation += f"Assistant: {text}\n"
        return conversation

    def evaluator(self, state: State) -> State:
        last_msg = state["messages"][-1]
        last_response = (
            (getattr(last_msg, "content", None) or "")
            or (state.get("final_worker_response") or "")
            or ""
        )

        system_message = """You are an evaluator that determines if a task has been completed successfully by an Assistant.
    Assess the Assistant's last response based on the given criteria. Respond with your feedback, and with your decision on whether the success criteria has been met,
    and whether more input is needed from the user."""

        user_message = f"""You are evaluating a conversation between the User and Assistant. You decide what action to take based on the last response from the Assistant.

    The entire conversation with the assistant, with the user's original request and all replies, is:
    {self.format_conversation(state["messages"])}

    The success criteria for this assignment is:
    {state["success_criteria"]}

    And the final response from the Assistant that you are evaluating is:
    {last_response}

    Respond with your feedback, and decide if the success criteria is met by this response.
    Also, decide if more user input is required, either because the assistant has a question, needs clarification, or seems to be stuck and unable to answer without help.

    The Assistant has access to a tool to write files. If the Assistant says they have written a file, then you can assume they have done so.
    Overall you should give the Assistant the benefit of the doubt if they say they've done something. But you should reject if you feel that more work should go into this.

    """
        if state["feedback_on_work"]:
            user_message += f"Also, note that in a prior attempt from the Assistant, you provided this feedback: {state['feedback_on_work']}\n"
            user_message += "If you're seeing the Assistant repeating the same mistakes, then consider responding that user input is required."

        evaluator_messages = [
            SystemMessage(content=system_message),
            HumanMessage(content=user_message),
        ]

        eval_result = self.evaluator_llm_with_output.invoke(evaluator_messages)
        new_state = {
            "messages": [
                {
                    "role": "assistant",
                    "content": f"Evaluator Feedback on this answer: {eval_result.feedback}",
                }
            ],
            "feedback_on_work": eval_result.feedback,
            "success_criteria_met": eval_result.success_criteria_met,
            "user_input_needed": eval_result.user_input_needed,
        }
        return new_state

    def route_based_on_evaluation(self, state: State) -> str:
        if state["success_criteria_met"] or state["user_input_needed"]:
            return "END"
        else:
            return "worker"

    async def build_graph(self):
        graph_builder = StateGraph(State)

        graph_builder.add_node("planner", self.planner)
        graph_builder.add_node("worker", self.worker)
        graph_builder.add_node("tools", ToolNode(tools=self.tools))
        graph_builder.add_node("evaluator", self.evaluator)

        graph_builder.add_edge(START, "planner")
        graph_builder.add_edge("planner", "worker")
        graph_builder.add_conditional_edges(
            "worker", self.worker_router, {"tools": "tools", "evaluator": "evaluator"}
        )
        graph_builder.add_edge("tools", "worker")
        graph_builder.add_conditional_edges(
            "evaluator", self.route_based_on_evaluation, {"worker": "worker", "END": END}
        )

        self.graph = graph_builder.compile(checkpointer=self.memory)

    async def run_superstep(self, message, success_criteria, clarifications, history):
        config = {"configurable": {"thread_id": self.sidekick_id}}

        parts = [message.strip()]
        if clarifications and str(clarifications).strip():
            parts.append(f"User clarifications:\n{str(clarifications).strip()}")
        if success_criteria and str(success_criteria).strip():
            parts.append(f"Success criteria:\n{str(success_criteria).strip()}")
        combined = "\n\n".join(parts)

        state: State = {
            "messages": [HumanMessage(content=combined)],
            "success_criteria": success_criteria or "The answer should be clear and accurate",
            "feedback_on_work": None,
            "success_criteria_met": False,
            "user_input_needed": False,
            "plan": None,
            "final_worker_response": None,
            "clarifications_text": (str(clarifications).strip() if clarifications else None),
        }
        result = await self.graph.ainvoke(state, config=config)

        user_text = message
        if clarifications and str(clarifications).strip():
            user_text = f"{message}\n\nClarifications: {clarifications.strip()}"
        user = {"role": "user", "content": user_text}
        assistant_text = result.get("final_worker_response") or "(no assistant text)"
        reply = {"role": "assistant", "content": assistant_text}
        feedback_text = result.get("feedback_on_work") or ""
        feedback = {"role": "assistant", "content": f"Evaluator feedback: {feedback_text}"}
        return history + [user, reply, feedback]

    async def aclose(self):
        """Close SQLite checkpointer and browser resources."""
        if self.memory_ctx is not None:
            try:
                await self.memory_ctx.__aexit__(None, None, None)
            except Exception:
                pass
            self.memory_ctx = None
            self.memory = None
        if self.browser:
            try:
                await self.browser.close()
            except Exception:
                pass
            self.browser = None
        if self.playwright:
            try:
                await self.playwright.stop()
            except Exception:
                pass
            self.playwright = None

    def cleanup(self):
        """Sync cleanup for Gradio delete_callback (browser only; SQL closed via aclose on reset)."""
        if self.browser:
            try:
                loop = asyncio.get_running_loop()
                loop.create_task(self.browser.close())
                if self.playwright:
                    loop.create_task(self.playwright.stop())
            except RuntimeError:
                asyncio.run(self.browser.close())
                if self.playwright:
                    asyncio.run(self.playwright.stop())
            self.browser = None
            self.playwright = None

In [ ]:
async def setup():
    sidekick = Sidekick()
    await sidekick.setup()
    return sidekick


async def get_clarifying_questions(message, success_criteria):
    """Generate exactly 3 clarifying questions from the user's request."""
    if not message or not str(message).strip():
        return " Enter a request first, then click again. "
    llm = ChatOpenAI(model=MODEL, base_url=openrouter_base_url, api_key=openrouter_api_key).with_structured_output(ClarifyingQuestions)
    prompt = (
        f"User request:\n{message}\n\nSuccess criteria:\n{success_criteria or '(none given)'}\n\n"
        "Generate exactly 3 short clarifying questions (scope, depth, constraints)."
    )
    out = llm.invoke([HumanMessage(content=prompt)])
    return "### Clarifying questions\n" + "\n".join(f"{i + 1}. {q}" for i, q in enumerate(out.questions))


async def process_message(sidekick, message, success_criteria, clarifications, history):
    if sidekick is None:
        return history + [{"role": "assistant", "content": "Sidekick failed to initialize. Please reset."}], None
    results = await sidekick.run_superstep(message, success_criteria, clarifications, history)
    return results, sidekick

In [39]:
async def reset(sidekick):
    if sidekick is not None:
        await sidekick.aclose()
    new_sidekick = Sidekick()
    await new_sidekick.setup()
    return "", "", "", "", [], new_sidekick


def free_resources(sidekick):
    print("Cleaning up")
    try:
        if sidekick:
            sidekick.cleanup()
    except Exception as e:
        print(f"Exception during cleanup: {e}")

### Display with Gradio!

In [ ]:
with gr.Blocks(title="Sidekick", theme=gr.themes.Default(primary_hue="emerald")) as demo:
    gr.Markdown("## Upgraded Sidekick Personal Co-Worker")
    gr.Markdown(
        "1. Enter your request and success criteria. \n"
        "2. **Get clarifying questions** (3 questions). \n"
        "3. Answer them in **Clarifications** (optional but recommended). \n"
        "4. **Go!** runs planner agent → worker agent → tools → evaluator (with SQL checkpoint memory)."
    )
    sidekick = gr.State(delete_callback=free_resources)

    with gr.Row():
        chatbot = gr.Chatbot(label="Sidekick", height=300, type="messages")
    with gr.Group():
        with gr.Row():
            message = gr.Textbox(show_label=False, placeholder="Your request to the Sidekick")
        with gr.Row():
            success_criteria = gr.Textbox(
                show_label=False, placeholder="What are your success criteria?"
            )
        with gr.Row():
            questions_md = gr.Markdown("*Click 'Get clarifying questions' to see 3 questions.*")
        with gr.Row():
            clarifications = gr.Textbox(
                show_label=False,
                placeholder="Your answers to the clarifying questions (used to tune the run)",
                lines=3,
            )
    with gr.Row():
        questions_button = gr.Button("Get clarifying questions", variant="secondary")
        go_button = gr.Button("Go!", variant="primary")
        reset_button = gr.Button("Reset", variant="stop")

    demo.load(setup, [], [sidekick])
    questions_button.click(
        get_clarifying_questions, [message, success_criteria], [questions_md]
    )
    message.submit(
        process_message, [sidekick, message, success_criteria, clarifications, chatbot], [chatbot, sidekick]
    )
    success_criteria.submit(
        process_message, [sidekick, message, success_criteria, clarifications, chatbot], [chatbot, sidekick]
    )
    go_button.click(
        process_message, [sidekick, message, success_criteria, clarifications, chatbot], [chatbot, sidekick]
    )
    reset_button.click(
        reset, [sidekick], [message, success_criteria, clarifications, questions_md, chatbot, sidekick]
    )


demo.launch(inbrowser=True)
